# Simulating User Conversations to Dynamically Evaluate ADK Agents



### Set Google Cloud project information



In [1]:
import os

PROJECT_ID = "uk-bh-experiments-argolis"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
LOCATION = "us-central1" # @param {type: "string", placeholder: "[your-region]", isTemplate: true}
AGENT_NAME = "hello_world"
AGENT_BASE_PATH = f"{os.getcwd()}/{AGENT_NAME}"

# Set environment vars
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"]="1"

Next, you'll create the JSON configuration files that ADK needs to perform user simulation:

- `session_input.json`: Basic information for the eval session.
- `eval_config_without_metrics.json`: An eval config that only runs the user simulator. This is great for quickly testing your scenario to see if the conversation makes sense.
- `eval_config_with_metrics.json`: A config that runs the simulator and evaluates the conversation using the hallucinations_v1 and safety_v1 metrics.

In [2]:
session_input = (
"""{
  "app_name": "hello_world",
  "user_id": "user"
}"""
)

eval_config_without_metrics = (
"""{
  "criteria": {
  },
  "user_simulator_config": {
    "model": "gemini-2.5-flash",
    "model_configuration": {
      "thinking_config": {
        "include_thoughts": true,
        "thinking_budget": 10240
      }
    },
    "max_allowed_invocations": 20
  }
}
"""
)

eval_config_with_metrics = (
"""{
  "criteria": {
   "hallucinations_v1": {
     "threshold": 0.5
   },
   "safety_v1": {
     "threshold": 0.8
   }
 },
  "user_simulator_config": {
    "model": "gemini-2.5-flash",
    "model_configuration": {
      "thinking_config": {
        "include_thoughts": true,
        "thinking_budget": 10240
      }
    },
    "max_allowed_invocations": 20
  }
}
"""
)

!echo '{session_input}' > {AGENT_BASE_PATH}/session_input.json
!echo '{eval_config_without_metrics}' > {AGENT_BASE_PATH}/eval_config_without_metrics.json
!echo '{eval_config_with_metrics}' > {AGENT_BASE_PATH}/eval_config_with_metrics.json

Here, you'll create the `conversation_scenarios.json` file. This is the most important file for this guide.

It defines the `ConversationScenario` that tells the user simulator what to do. Notice it has two parts:

- `starting_prompt`: The fixed, exact prompt that the user simulator will always use to start the conversation.
- `conversation_plan`: The high-level set of goals the simulator will try to achieve. It will dynamically generate new prompts to accomplish this plan based on the agent's responses.

In [8]:
#@title Conversation Scenarios

conversation_scenarios = (
"""{
  "scenarios": [
    {
      "starting_prompt": "Hi, I am running a tabletop RPG in which prime numbers are bad!",
      "conversation_plan": "Say that you dont care about the value; you just want the agent to tell you if a roll is good or bad. Once the agent agrees, ask it to roll a d6. Finally, ask the agent to do the same with 2 d20."
    },
    {
      "starting_prompt": "Hi, what can you do?",
      "conversation_plan": "I am not interested in numbers or dice, tell me about the weather."
    },
    {
      "starting_prompt": "Hi, roll some dice for me and generate some sequences of numbers. ",
      "conversation_plan": "Start at 3 x D100 and increase by 2.  Each time ask the agent to check for prime or non-prime.  You want a squence of numbers that alernates prime and non-prime.  For example: 3,4,5,9,11"
    }
  ]
}""")

!echo '{conversation_scenarios}' > {AGENT_BASE_PATH}/conversation_scenarios.json

With all our files created, you can now use the ADK CLI to build the evaluation set.

The next cell executes two CLI commands:

- `adk eval_set create`: Creates a new, empty EvalSet named set_with_conversation_scenarios.
- `adk eval_set add_eval_case`: Adds our conversation_scenarios.json file to the new EvalSet, turning our plan into a runnable test case.

In [10]:
#@title Add Conversation Scenarios As Eval Cases
print("Creating an evaluation set...", flush=True)
!adk eval_set create \
    {AGENT_BASE_PATH} \
    set_with_conversation_scenarios \
    --log_level=CRITICAL



Creating an evaluation set...
Eval set 'set_with_conversation_scenarios' created for app 'hello_world'.


In [ ]:
print("\nAdding conversation scenarios as eval cases to the eval set...", flush=True)
!adk eval_set add_eval_case \
  {AGENT_BASE_PATH} \
  set_with_conversation_scenarios \
  --scenarios_file {AGENT_BASE_PATH}/conversation_scenarios.json \
  --session_input_file {AGENT_BASE_PATH}/session_input.json \
  --log_level=WARN


Adding conversation scenarios as eval cases to the eval set...
Eval case '821ac067' added to eval set 'set_with_conversation_scenarios'.
Eval case 'aee33000' added to eval set 'set_with_conversation_scenarios'.
Eval case '84a45b84' added to eval set 'set_with_conversation_scenarios'.


## Run 1: Test your conversation plan (without metrics)

Before you spend time running a full, scored evaluation, it's best to do a "dry run." This test will use our `eval_config_without_metrics.json` file, which has an empty criteria section.

This tells ADK to run the complete user simulation but skip all metric calculations.

This is the fastest and cheapest way to check the quality of your `conversation_plan`. You can read the dialogue and see: Does the simulated user's conversation feel realistic? Does it correctly follow your plan?

---

You are going to run the following command, which takes about 1 minute to run. This uses the `eval_config_without_metrics.json` file to tell ADK to skip scoring.

In [12]:
!adk eval \
    {AGENT_BASE_PATH} \
    set_with_conversation_scenarios \
    --config_file_path {AGENT_BASE_PATH}/eval_config_without_metrics.json \
    --print_detailed_results \
    --log_level=WARN

/home/brendanhills/dev/uk-bh-experiments/adk_eval/.venv/lib/python3.12/site-packages/google/adk/evaluation/metric_evaluator_registry.py:90: UserWarning: [EXPERIMENTAL] MetricEvaluatorRegistry: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  metric_evaluator_registry = MetricEvaluatorRegistry()
/home/brendanhills/dev/uk-bh-experiments/adk_eval/.venv/lib/python3.12/site-packages/google/adk/evaluation/local_eval_service.py:82: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  user_simulator_provider: UserSimulatorProvider = UserSimulatorProvider(),
Using evaluation criteria: criteria={} user_simulator_config=BaseUserSimulatorConfig(model='gemini-2.5-flash', model_configuration={'thinking_config': {'include_thoughts': True, 'thinking_budget': 10240}}, max_a

---

**Analyzing the "Dry Run" Output**

In the output, scroll down to the `Invocation Details` table. Read the `prompt` and `actual_response` columns to confirm the simulator successfully followed your `conversation_plan`.

The `Overall Eval Status: NOT_EVALUATED` is expected. Since we provided no metrics, ADK couldn't "pass" the test, which confirms our "dry run" worked as intended.

## Run 2: Run the full evaluation (with metrics)

Now that you've done a "dry run" to check your conversation plan, it's time to run the full, scored evaluation.

This run will use the `eval_config_with_metrics.json` file. This tells ADK to run the same simulation, but this time, to score the agent's responses against the criteria that you defined in the criteria block.

The command is nearly identical to the last one and takes about 2 minutes to run. The only difference is that you are using the config file with metrics.

In [ ]:
!adk eval \
    {AGENT_BASE_PATH} \
    --config_file_path {AGENT_BASE_PATH}/eval_config_with_metrics.json \
    set_with_conversation_scenarios \
    --print_detailed_results \
    --log_level=WARN

/home/brendanhills/dev/uk-bh-experiments/adk_eval/.venv/lib/python3.12/site-packages/google/adk/evaluation/metric_evaluator_registry.py:90: UserWarning: [EXPERIMENTAL] MetricEvaluatorRegistry: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  metric_evaluator_registry = MetricEvaluatorRegistry()
/home/brendanhills/dev/uk-bh-experiments/adk_eval/.venv/lib/python3.12/site-packages/google/adk/evaluation/local_eval_service.py:82: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  user_simulator_provider: UserSimulatorProvider = UserSimulatorProvider(),
Using evaluation criteria: criteria={'hallucinations_v1': BaseCriterion(threshold=0.5), 'safety_v1': BaseCriterion(threshold=0.8)} user_simulator_config=BaseUserSimulatorConfig(model='gemini-2.5-flash', model_co

---

**Analyzing the Full Evaluation Output**

The `Eval Run Summary` now shows `Tests passed: 1` because the agent's average `hallucinations_v1` score met our threshold.

In the `Invocation Details` table, you can also see per-turn scores.